# 순환 신경망

이전 모듈에서는 텍스트의 풍부한 의미 표현에 대해 다뤘습니다. 우리가 사용해온 아키텍처는 문장 내 단어들의 집합적 의미를 포착하지만, 단어들의 <strong>순서</strong>는 고려하지 않습니다. 왜냐하면 임베딩 다음에 수행되는 집합 연산이 원래 텍스트에서 이 정보를 제거하기 때문입니다. 이러한 모델들은 단어 순서를 표현할 수 없기 때문에 텍스트 생성이나 질문 응답과 같은 더 복잡하거나 애매한 작업을 해결할 수 없습니다.

텍스트 시퀀스의 의미를 포착하기 위해, 우리는 **순환 신경망**(recurrent neural network, RNN)이라는 신경망 아키텍처를 사용할 것입니다. RNN을 사용할 때는 문장을 한 번에 하나의 토큰씩 네트워크에 입력하고, 네트워크는 일부 **상태(state)** 를 생성하며, 이 상태를 다음 토큰과 함께 다시 네트워크에 전달합니다.

![순환 신경망의 예시 생성을 보여주는 이미지.](../../../../../translated_images/ko/rnn.27f5c29c53d727b5.webp)

입력 토큰 시퀀스 $X_0,\dots,X_n$가 주어지면, RNN은 신경망 블록의 시퀀스를 생성하고, 역전파를 사용하여 이 시퀀스를 종단 간(end-to-end)으로 학습합니다. 각 네트워크 블록은 $(X_i,S_i)$ 쌍을 입력으로 받아 $S_{i+1}$을 결과로 생성합니다. 최종 상태 $S_n$ 또는 출력 $Y_n$은 결과를 생성하기 위해 선형 분류기에 들어갑니다. 모든 네트워크 블록은 동일한 가중치를 공유하며, 하나의 역전파 과정을 통해 종단 간으로 학습됩니다.

> 위 그림은 풀어 쓴 순환 신경망 형태(왼쪽)와 더 컴팩트한 순환 표현(오른쪽)을 보여줍니다. 모든 RNN 셀은 동일한 <strong>공유 가능한 가중치</strong>를 가진다는 점을 이해하는 것이 중요합니다.

상태 벡터 $S_0,\dots,S_n$가 네트워크를 통해 전달되기 때문에, RNN은 단어들 간의 순차적 의존성을 학습할 수 있습니다. 예를 들어, 시퀀스 중 어딘가에 단어 <em>not</em>이 등장하면, 상태 벡터 내의 특정 요소들을 부정하는 법을 학습할 수 있습니다.

내부적으로 각 RNN 셀은 두 개의 가중치 행렬 $W_H$와 $W_I$, 그리고 편향 $b$를 포함합니다. 각 RNN 단계에서 입력 $X_i$와 입력 상태 $S_i$가 주어지면, 출력 상태는 $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$로 계산되며, 여기서 $f$는 활성화 함수(보통은 $\tanh$)입니다.

> 텍스트 생성(다음 단원에서 다룰 예정)이나 기계 번역 같은 문제에서는 각 RNN 단계에서 출력 값을 얻고자 합니다. 이 경우, 또 다른 행렬 $W_O$가 있고 출력은 $Y_i=f(W_O\times S_i+b_O)$로 계산됩니다.

이제 순환 신경망이 뉴스 데이터셋 분류에 어떻게 도움이 되는지 살펴봅시다.

> 샌드박스 환경에서는 필수 라이브러리를 설치하고 데이터를 미리 가져오기 위해 다음 셀을 실행해야 합니다. 로컬에서 실행 중이라면 다음 셀을 건너뛸 수 있습니다.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

대형 모델을 훈련할 때 GPU 메모리 할당이 문제가 될 수 있습니다. 또한, 데이터가 GPU 메모리에 맞으면서도 훈련 속도가 충분히 빠르도록 다양한 미니배치 크기를 실험해 볼 필요가 있습니다. 이 코드를 개인 GPU 머신에서 실행 중이라면, 훈련 속도를 높이기 위해 미니배치 크기를 조정해 볼 수 있습니다.

> **Note**: 특정 NVidia 드라이버 버전은 모델 훈련 후 메모리를 해제하지 않는 것으로 알려져 있습니다. 이 노트북에서 여러 예제를 실행하고 있으며, 특히 같은 노트북에서 직접 실험하는 경우 특정 환경에서 메모리가 부족해질 수 있습니다. 모델 훈련을 시작할 때 이상한 오류가 발생하면 노트북 커널을 재시작해 보시기 바랍니다.


In [3]:
batch_size = 16
embed_size = 64

## 간단한 RNN 분류기

간단한 RNN의 경우, 각 순환 유닛은 입력 벡터와 상태 벡터를 받아 새로운 상태 벡터를 생성하는 간단한 선형 네트워크입니다. Keras에서는 이를 `SimpleRNN` 계층으로 표현할 수 있습니다.

원-핫 인코딩된 토큰을 RNN 계층에 직접 전달할 수 있지만, 그 차원이 매우 높기 때문에 좋은 방법은 아닙니다. 따라서 임베딩 계층을 사용하여 단어 벡터의 차원을 낮추고, 뒤이어 RNN 계층을 쌓은 후, 마지막으로 `Dense` 분류기를 사용합니다.

> <strong>참고</strong>: 차원이 그리 높지 않은 경우, 예를 들어 문자 단위 토크나이징을 사용할 때는 원-핫 인코딩된 토큰을 RNN 셀에 직접 전달하는 것이 합리적일 수 있습니다.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **참고:** 여기서는 간단하게 하기 위해 훈련되지 않은 임베딩 레이어를 사용하지만, 더 나은 결과를 위해 이전 단원에서 설명한 대로 Word2Vec를 이용한 사전 훈련된 임베딩 레이어를 사용할 수 있습니다. 이 코드를 사전 훈련된 임베딩과 함께 작동하도록 수정하는 것은 좋은 연습이 될 것입니다.

이제 RNN을 훈련시켜 보겠습니다. 일반적으로 RNN은 훈련하기가 꽤 어렵습니다. 왜냐하면 RNN 셀을 시퀀스 길이만큼 펼치게 되면, 역전파에 관여하는 레이어 수가 매우 많아지기 때문입니다. 따라서 더 작은 학습률을 선택하고 더 큰 데이터셋으로 네트워크를 훈련시켜야 좋은 결과를 얻을 수 있습니다. 이 과정은 꽤 오랜 시간이 걸릴 수 있으므로 GPU 사용이 권장됩니다.

속도를 높이기 위해 설명은 제외하고 뉴스 제목에 대해서만 RNN 모델을 훈련시킬 것입니다. 설명까지 포함하여 훈련시키면 모델이 학습되는지 확인해 보실 수 있습니다.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> <strong>참고</strong> 정확도가 여기서는 낮을 수 있는데, 뉴스 제목만을 가지고 학습하기 때문입니다.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


이제 마스킹을 사용하고 있으므로 제목과 설명의 전체 데이터 세트에 대해 모델을 학습시킬 수 있습니다.

> **Note**: 뉴스 제목에 대해 학습된 벡터라이저를 사용하고 전체 기사 본문에 대해서는 사용하지 않았다는 것을 눈치채셨나요? 잠재적으로 일부 토큰이 무시될 수 있으므로 벡터라이저를 다시 학습하는 것이 좋습니다. 그러나 이것은 매우 작은 영향만 미칠 수 있으므로 단순함을 위해 이전에 학습된 벡터라이저를 사용하겠습니다.


## LSTM: 장기 단기 기억

RNN의 주요 문제 중 하나는 <strong>기울기 소실</strong>입니다. RNN은 상당히 길어질 수 있으며 역전파 과정에서 기울기를 네트워크의 첫 번째 층까지 모두 전달하는 데 어려움을 겪을 수 있습니다. 이런 경우, 네트워크는 멀리 떨어진 토큰들 간의 관계를 학습할 수 없습니다. 이 문제를 피하는 한 가지 방법은 <strong>게이트</strong>를 사용하여 **명시적인 상태 관리를** 도입하는 것입니다. 게이트를 도입하는 두 가지 가장 일반적인 아키텍처는 **장기 단기 기억**(LSTM)과 **게이트 릴레이 유닛**(GRU)입니다. 여기서는 LSTM을 다룹니다.

![장기 단기 기억 셀의 예를 보여주는 이미지](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

LSTM 네트워크는 RNN과 유사하게 구성되지만, 레이어 간에 전달되는 두 가지 상태가 있습니다: 실제 상태 $c$와 은닉 벡터 $h$입니다. 각 유닛에서 은닉 벡터 $h_{t-1}$는 입력 $x_t$와 결합되어 함께 <strong>게이트</strong>를 통해 상태 $c_t$와 출력 $h_{t}$에 어떤 영향이 있을지 제어합니다. 각 게이트는 시그모이드 활성화 함수를 사용하여 출력 값이 $[0,1]$ 범위에 있으며, 상태 벡터에 곱해질 때 비트 단위 마스크로 생각할 수 있습니다. LSTM에는 다음과 같은 게이트들이 있습니다(위 그림의 왼쪽부터 오른쪽 순서대로):
* <strong>망각 게이트</strong>는 벡터 $c_{t-1}$의 어떤 구성 요소를 잊어버려야 하는지, 어떤 것을 통과시켜야 하는지 결정합니다.
* <strong>입력 게이트</strong>는 입력 벡터와 이전 은닉 벡터에서 얼마나 많은 정보를 상태 벡터에 포함할지 결정합니다.
* <strong>출력 게이트</strong>는 새 상태 벡터를 가져와서 그 구성 요소 중 어느 것을 새로운 은닉 벡터 $h_t$를 생성하는 데 사용할지 결정합니다.

상태 $c$의 구성 요소들은 켜고 끌 수 있는 플래그로 생각할 수 있습니다. 예를 들어, 시퀀스에서 <em>Alice</em>라는 이름을 만났을 때, 우리는 이것이 여성임을 가리킨다고 추측하고 문장에 여성 명사가 있다는 플래그를 상태에서 올립니다. 그리고 <em>and Tom</em>이라는 단어들을 추가로 만나면 복수 명사라는 플래그를 올립니다. 이렇게 상태를 조작함으로써 문장의 문법적 특성을 추적할 수 있습니다.

> <strong>노트</strong>: LSTM 내부 구조를 이해하는 데 훌륭한 자료가 여기 있습니다: Christopher Olah의 [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).

LSTM 셀의 내부 구조가 복잡해 보일 수 있지만, Keras는 이를 `LSTM` 레이어 안에 숨기기 때문에 위 예제에서 해야 할 일은 순환 레이어만 교체하는 것입니다.


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> <strong>참고</strong>로, LSTM을 훈련시키는 것도 상당히 느리며, 훈련 초기에 정확도가 크게 증가하지 않는 것처럼 보일 수 있습니다. 좋은 정확도를 얻기 위해서는 어느 정도 기간 동안 훈련을 계속해야 할 수도 있습니다.


## 양방향 및 다층 RNN

지금까지의 예에서 순환 신경망은 시퀀스의 시작부터 끝까지 작동했습니다. 이는 우리가 읽거나 말을 듣는 방향과 같기 때문에 자연스럽게 느껴집니다. 그러나 입력 시퀀스에 대한 임의 접근이 필요한 경우에는 순환 계산을 양방향으로 실행하는 것이 더 의미가 있습니다. 양방향 계산을 허용하는 RNN을 <strong>양방향</strong> RNN이라고 하며, 순환 층을 특별한 `Bidirectonal` 층으로 감싸서 만들 수 있습니다.

> <strong>참고</strong>: `Bidirectional` 층은 그 안에 있는 층을 두 개 복제하고, 그 중 하나의 `go_backwards` 속성을 `True`로 설정하여 시퀀스의 반대 방향으로 이동하도록 만듭니다.

단방향이든 양방향이든 순환 신경망은 시퀀스 내에서 패턴을 포착하여 상태 벡터에 저장하거나 출력으로 반환합니다. 합성곱 신경망과 마찬가지로, 첫 번째 층 다음에 또 다른 순환 층을 쌓아 첫 층이 추출한 낮은 수준의 패턴으로부터 더 높은 수준의 패턴을 포착할 수 있습니다. 이를 통해 이전 층의 출력을 다음 층의 입력으로 전달하는 두 개 이상의 순환 신경망으로 구성된 **다층 RNN** 개념이 등장합니다.

![Multilayer long-short-term-memory- RNN을 보여주는 이미지](../../../../../translated_images/ko/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Fernando López의 [이 멋진 글](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3)에서 가져온 사진입니다.*

Keras는 이러한 네트워크를 구성하는 작업을 쉽게 만드는데, 모델에 순환 층을 더 추가하기만 하면 됩니다. 마지막 층을 제외한 모든 층에 대해 `return_sequences=True` 매개변수를 지정해야 하는데, 이는 순환 계산의 최종 상태뿐 아니라 모든 중간 상태를 반환하도록 층을 설정해야 하기 때문입니다.

분류 문제를 위한 두 층의 양방향 LSTM을 만들어 봅시다.

> <strong>참고</strong> 이 코드는 완료하는 데 시간이 꽤 오래 걸리지만 지금까지 본 것 중 가장 높은 정확도를 제공합니다. 그러니 기다리면서 결과를 보는 것도 가치가 있을 수 있습니다.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## 다른 작업을 위한 RNN

지금까지 우리는 RNN을 사용하여 텍스트 시퀀스를 분류하는 데 중점을 두었습니다. 그러나 RNN은 텍스트 생성 및 기계 번역과 같은 훨씬 더 많은 작업을 처리할 수 있습니다 — 다음 단원에서 이러한 작업들을 살펴보겠습니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
